# CRF for NER — Setup & Environment Check

We will reuse CoNLL-2003 from `data\conll2003`. This cell verifies the interpreter and ensures `sklearn-crfsuite` is installed for training a CRF tagger.


In [1]:
import sys, subprocess, importlib
from pathlib import Path

print("Python executable:", sys.executable)

# Ensure sklearn-crfsuite is available
try:
    import sklearn_crfsuite  # type: ignore
    print("sklearn-crfsuite already installed.")
except Exception:
    print("Installing sklearn-crfsuite ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "sklearn-crfsuite"])
    importlib.invalidate_caches()
    import sklearn_crfsuite  # type: ignore
    print("Installed sklearn-crfsuite.")

# Data base path (same as HMM notebook)
BASE = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003")
assert BASE.exists(), f"Expected data folder at {BASE}"
print("Data base path:", BASE)


Python executable: c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\Scripts\python.exe
Installing sklearn-crfsuite ...
Installed sklearn-crfsuite.
Data base path: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003


## Data loading — CoNLL-2003 (reuse path and format)

Parses sentences as lists of `(token, tag)` from:
`C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\`

Accepts common filenames:
- `eng.train`, `eng.testa`, `eng.testb` **or**
- `train.txt`, `valid.txt`/`dev.txt`, `test.txt`

Outputs: `train_sents`, `valid_sents`, `test_sents` + quick stats.


In [2]:
from pathlib import Path
from collections import Counter

BASE = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003")

def pick_existing(options):
    for name in options:
        p = BASE / name
        if p.exists():
            return p
    return None

TRAIN_FILE = pick_existing(["eng.train", "train.txt", "train"])
VALID_FILE = pick_existing(["eng.testa", "valid.txt", "dev.txt", "valid", "dev"])
TEST_FILE  = pick_existing(["eng.testb", "test.txt", "test"])

assert TRAIN_FILE and TRAIN_FILE.exists(), f"Train file not found under {BASE}"
assert VALID_FILE and VALID_FILE.exists(), f"Valid/dev file not found under {BASE}"
assert TEST_FILE  and TEST_FILE.exists(),  f"Test file not found under {BASE}"

def load_conll(path: Path):
    sents, sent = [], []
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line:
                if sent:
                    sents.append(sent)
                    sent = []
                continue
            if line.startswith("-DOCSTART-"):
                continue
            parts = line.split()
            token, tag = parts[0], parts[-1]
            sents.append(sent) if False else None  # no-op to keep structure visible
            sent.append((token, tag))
    if sent:
        sents.append(sent)
    return sents

train_sents = load_conll(TRAIN_FILE)
valid_sents = load_conll(VALID_FILE)
test_sents  = load_conll(TEST_FILE)

def stats(name, sents):
    n_sents = len(sents)
    n_toks  = sum(len(s) for s in sents)
    tags = Counter(tag for s in sents for _, tag in s)
    return name, n_sents, n_toks, len(tags), tags.most_common(5)

print("Files:")
print("  Train:", TRAIN_FILE)
print("  Valid:", VALID_FILE)
print("  Test :", TEST_FILE)
print("\nDataset stats (name, #sentences, #tokens, #unique_tags, top5_tags):")
for row in [stats("train", train_sents), stats("valid", valid_sents), stats("test", test_sents)]:
    print(" ", row)

print("\nPreview (train[0], first 12):")
for tok, tag in (train_sents[0][:12] if train_sents else []):
    print(f"{tok:15s} {tag}")
if train_sents and len(train_sents[0]) > 12:
    print("... (truncated)")


Files:
  Train: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\train.txt
  Valid: C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\valid.txt
  Test : C:\Users\murth\Desktop\nlpSession02\codeBase\data\conll2003\test.txt

Dataset stats (name, #sentences, #tokens, #unique_tags, top5_tags):
  ('train', 14041, 203621, 9, [('O', 169578), ('B-LOC', 7140), ('B-PER', 6600), ('B-ORG', 6321), ('I-PER', 4528)])
  ('valid', 3250, 51362, 9, [('O', 42759), ('B-PER', 1842), ('B-LOC', 1837), ('B-ORG', 1341), ('I-PER', 1307)])
  ('test', 3453, 46435, 9, [('O', 38323), ('B-LOC', 1668), ('B-ORG', 1661), ('B-PER', 1617), ('I-PER', 1156)])

Preview (train[0], first 12):
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O


## Feature extraction for CRF (word + shape + context ±2)

We build rich, **hand-crafted features** per token:

- **Lexical:** lowercased word, prefixes/suffixes (1–3 chars), is-initial-cap, is-all-caps, is-all-lower, is-digit, has-hyphen.
- **Shape:** simplified pattern (e.g., `Xxxx`, `xxx`, `XX`, `d`, `Xd`).
- **Context:** previous/next tokens (±1 and ±2) with the same feature set (prefixed with `-1:`, `+1:`, `-2:`, `+2:`).
- **Position flags:** `BOS`/`EOS` for sentence boundaries.

Outputs:
- `sent2features(sent)` → list of feature dicts (one per token)
- `sent2labels(sent)` → gold BIO tag sequence

We’ll use this to create `X_train/y_train`, `X_valid/y_valid`, `X_test/y_test`.


In [3]:
import re

def word_shape(token: str) -> str:
    """
    Map token to a coarse shape:
    - Uppercase letters -> 'X'
    - Lowercase letters -> 'x'
    - Digits -> 'd'
    - Other -> '-'
    Examples: 'U.N.' -> 'X-X-', 'London' -> 'Xxxxx', '1999' -> 'dddd'
    """
    out = []
    for ch in token:
        if ch.isupper():
            out.append('X')
        elif ch.islower():
            out.append('x')
        elif ch.isdigit():
            out.append('d')
        else:
            out.append('-')
    return ''.join(out)

def token_features(sent, i):
    """
    Features for token at index i in a sentence of (token, tag) pairs.
    """
    token = sent[i][0]
    lower = token.lower()

    feats = {
        "bias": 1.0,  # constant feature
        "word.lower": lower,
        "word.isupper": token.isupper(),
        "word.istitle": token.istitle(),
        "word.islower": token.islower(),
        "word.isdigit": token.isdigit(),
        "word.has-hyphen": "-" in token,
        "word.isalnum": token.isalnum(),
        "word.shape": word_shape(token),
        "pref1": lower[:1],
        "pref2": lower[:2] if len(lower) >= 2 else lower,
        "pref3": lower[:3] if len(lower) >= 3 else lower,
        "suf1": lower[-1:],
        "suf2": lower[-2:] if len(lower) >= 2 else lower,
        "suf3": lower[-3:] if len(lower) >= 3 else lower,
    }

    # Previous tokens (context)
    if i > 0:
        prev = sent[i-1][0]
        prev_l = prev.lower()
        feats.update({
            "-1:word.lower": prev_l,
            "-1:istitle": prev.istitle(),
            "-1:isupper": prev.isupper(),
            "-1:shape": word_shape(prev),
            "-1:suf2": prev_l[-2:] if len(prev_l) >= 2 else prev_l,
            "-1:pref2": prev_l[:2] if len(prev_l) >= 2 else prev_l,
        })
    else:
        feats["BOS"] = True

    if i > 1:
        prev2 = sent[i-2][0]
        prev2_l = prev2.lower()
        feats.update({
            "-2:word.lower": prev2_l,
            "-2:shape": word_shape(prev2),
        })

    # Next tokens (context)
    if i < len(sent) - 1:
        nxt = sent[i+1][0]
        nxt_l = nxt.lower()
        feats.update({
            "+1:word.lower": nxt_l,
            "+1:istitle": nxt.istitle(),
            "+1:isupper": nxt.isupper(),
            "+1:shape": word_shape(nxt),
            "+1:suf2": nxt_l[-2:] if len(nxt_l) >= 2 else nxt_l,
            "+1:pref2": nxt_l[:2] if len(nxt_l) >= 2 else nxt_l,
        })
    else:
        feats["EOS"] = True

    if i < len(sent) - 2:
        nxt2 = sent[i+2][0]
        nxt2_l = nxt2.lower()
        feats.update({
            "+2:word.lower": nxt2_l,
            "+2:shape": word_shape(nxt2),
        })

    return feats

def sent2features(sent):
    return [token_features(sent, i) for i in range(len(sent))]

def sent2labels(sent):
    return [tag for _, tag in sent]

# Smoke test on first training sentence
X_train_sample = sent2features(train_sents[0])
y_train_sample = sent2labels(train_sents[0])
print(f"Sample features for token 0:\n{X_train_sample[0]}")
print(f"\nSample labels (first 10):\n{y_train_sample[:10]}")


Sample features for token 0:
{'bias': 1.0, 'word.lower': 'eu', 'word.isupper': True, 'word.istitle': False, 'word.islower': False, 'word.isdigit': False, 'word.has-hyphen': False, 'word.isalnum': True, 'word.shape': 'XX', 'pref1': 'e', 'pref2': 'eu', 'pref3': 'eu', 'suf1': 'u', 'suf2': 'eu', 'suf3': 'eu', 'BOS': True, '+1:word.lower': 'rejects', '+1:istitle': False, '+1:isupper': False, '+1:shape': 'xxxxxxx', '+1:suf2': 'ts', '+1:pref2': 're', '+2:word.lower': 'german', '+2:shape': 'Xxxxxx'}

Sample labels (first 10):
['B-ORG', 'O', 'B-MISC', 'O', 'O', 'O', 'B-MISC', 'O', 'O']
